<a href="https://colab.research.google.com/github/gautamthampy/CMPE256-Group10/blob/baseline-covisitation/RecSysProjject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Do not consider this code as part of the submission

In [ ]:
# 1. Setup & imports

import math
import random
from collections import Counter, defaultdict
import numpy as np

In [ ]:
# 2. Load data

data = {}

file_path = "/content/train-2.txt"

with open(file_path, "r") as f:
    for line in f:
        parts = line.strip().split()
        if not parts:
            continue
        user = parts[0]
        items = parts[1:]
        data[user] = items

print("Number of users:", len(data))

In [ ]:
from collections import Counter

item_counts = Counter()
for items in data.values():
    item_counts.update(items)

print("Total unique items:", len(item_counts))
print("Example user, items:", next(iter(data.items())))

In [ ]:
# 3. Build train/validation split

random.seed(42)

train_data = {}  # user -> list of items (without held out)
val_items   = {}  # user -> single held-out item (str)

for user, items in data.items():
    if len(items) == 0:
        continue
    held_out = random.choice(items)
    remaining = [it for it in items if it != held_out]
    # if user had duplicates of held_out, they all get removed; fine for implicit data

    train_data[user] = remaining
    val_items[user] = held_out

print("Users in train_data:", len(train_data))
print("Users in val_items:", len(val_items))

In [ ]:
# 4. NDCG@K for single held-out item per user

def ndcg_at_k_single(recommended, true_item, k=20):
    """
    recommended: list of item ids, ranked from best to worst
    true_item: the held-out item id (string)
    """
    try:
        rank = recommended.index(true_item)
    except ValueError:
        return 0.0

    if rank >= k:
        return 0.0

    # DCG with relevance 1 at position rank
    return 1.0 / math.log2(rank + 2)  # +2 because ranks are 0-based


def mean_ndcg_at_k(recommender_fn, users, val_dict, k=20):
    """
    recommender_fn(user, k) -> list of item_ids
    users: iterable of user ids
    val_dict: dict user -> true_item
    """
    scores = []
    for u in users:
        true_item = val_dict[u]
        recs = recommender_fn(u, k=k)
        scores.append(ndcg_at_k_single(recs, true_item, k))
    return sum(scores) / len(scores)

Algorithm 1: The popularity-based recommender serves as our simplest baseline model. It ranks all items globally by how frequently they appear in the training data and recommends the top-20 most popular items that a user has not already interacted with. Because it does not use any personalization or user-specific signals, every user receives nearly the same recommendations aside from items they have already seen. While extremely fast and easy to implement, this approach performs poorly on ranking metrics, as it generally fails to retrieve each user’s unique held-out item. In our leave-one-out evaluation, the popularity model achieved an NDCG@20 of 0.0047, confirming that


In [ ]:
# 5. Popularity baseline

pop_counts = Counter()
for items in train_data.values():
    pop_counts.update(items)

print("Unique items in train:", len(pop_counts))
print("Top 5 most popular items:", pop_counts.most_common(5))

# Global ranked list of items by popularity
items_by_pop = [it for it, _ in pop_counts.most_common()]

In [ ]:
def recommend_popularity(user, k=20):
    """
    Recommend top-k most popular items the user has NOT seen in train_data.
    """
    seen = set(train_data[user])  # items user has interacted with in train set
    recs = []
    for it in items_by_pop:
        if it in seen:
            continue
        recs.append(it)
        if len(recs) == k:
            break
    return recs

In [ ]:
users_list = list(train_data.keys())

pop_ndcg_20 = mean_ndcg_at_k(recommend_popularity, users_list, val_items, k=20)
print("Popularity NDCG@20:", pop_ndcg_20)

Algorithm 2: The co-visitation model is an item-based collaborative filtering approach that recommends items frequently co-occurring with those a user has already interacted with. We build a co-occurrence matrix by counting how often pairs of items appear together across users, applying trimming and neighbor limits to keep the model efficient. For a given user, candidate items are scored based on how strongly they co-occur with the user’s seen items, with a normalization step to reduce popularity bias. This model captures item–item relationships and provides personalized recommendations based on users’ historical interactions. In evaluation, the co-visitation approach significantly outperformed the popularity baseline, achieving an NDCG@20 of approximately 0.083, indicating meaningful improvements in ranking the user’s held-out item.

In [ ]:
# 6. Co-visitation model (item-based)

def build_covisit_model(train_data, max_items_per_user=80, max_neighbors=400):
    """
    Returns:
      cooc: dict item_i -> dict item_j -> cooc_count
      pop_counts: Counter of item frequencies in train_data
    """
    # global popularity for trimming
    pop_counts = Counter()
    for items in train_data.values():
        pop_counts.update(items)

    # sort items by global popularity for each user and trim
    pop_rank = {item: rank for rank, (item, _) in enumerate(pop_counts.most_common())}

    trimmed_users = {}
    for user, items in train_data.items():
        # sort this user's items by popularity (most popular first)
        sorted_items = sorted(items, key=lambda it: pop_rank[it])
        trimmed_users[user] = sorted_items[:max_items_per_user]

    cooc = defaultdict(lambda: defaultdict(int))

    # build co-occurrence counts
    for items in trimmed_users.values():
        uniq = list(set(items))
        n = len(uniq)
        for i_idx in range(n):
            for j_idx in range(i_idx + 1, n):
                i = uniq[i_idx]
                j = uniq[j_idx]
                cooc[i][j] += 1
                cooc[j][i] += 1

    # prune neighbors
    for i, neighbors in list(cooc.items()):
        if len(neighbors) > max_neighbors:
            # keep only the most co-visited neighbors
            top_neighbors = sorted(neighbors.items(), key=lambda x: x[1], reverse=True)[:max_neighbors]
            cooc[i] = dict(top_neighbors)

    return cooc, pop_counts

In [ ]:
import json

def fix_github_rendering_error(input_file, output_file):
    """
    Removes the 'widgets' entry from the notebook metadata to fix
    GitHub rendering errors.
    """
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            notebook = json.load(f)

        if 'metadata' in notebook and 'widgets' in notebook['metadata']:
            print(f"Found 'widgets' metadata in {input_file}. Removing it...")
            del notebook['metadata']['widgets']

            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(notebook, f, indent=1)
            print(f"Success! Fixed notebook saved as: {output_file}")
        else:
            print("No 'widgets' metadata found. The file might already be clean.")

    except FileNotFoundError:
        print(f"Error: File '{input_file}' not found. Please upload it to the Files sidebar.")

# Example usage:
# 1. Upload your .ipynb file to the left sidebar
# 2. Update the filename below
# fix_github_rendering_error('MyNotebook.ipynb', 'MyNotebook_Fixed.ipynb')

In [ ]:
cooc, pop_counts_covisit = build_covisit_model(train_data,
                                              max_items_per_user=80,
                                              max_neighbors=400)
print("Co-visitation model built. Example item neighbors:")
example_item = next(iter(cooc.keys()))
print("Item:", example_item, "neighbors count:", len(cooc[example_item]))

In [ ]:
def recommend_covisit(user, k=20):
    """
    Recommend items using co-visitation model for this user.
    """
    seen = set(train_data[user])
    scores = defaultdict(float)

    for i in seen:
        if i not in cooc:
            continue
        for j, c in cooc[i].items():
            if j in seen:
                continue
            # normalized co-occurrence
            scores[j] += c / math.sqrt(pop_counts_covisit[i] * pop_counts_covisit[j])

    if not scores:
        # fallback to popularity baseline
        return recommend_popularity(user, k=k)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    recs = [it for it, _ in ranked[:k]]

    # If fewer than k recs, pad using popularity
    if len(recs) < k:
        extra = recommend_popularity(user, k=k + 50)
        for it in extra:
            if it not in seen and it not in recs:
                recs.append(it)
                if len(recs) == k:
                    break

    return recs

In [ ]:
covisit_ndcg_20 = mean_ndcg_at_k(recommend_covisit, users_list, val_items, k=20)
print("Co-visitation NDCG@20:", covisit_ndcg_20)

In [ ]:
# 7. Build final co-visitation model on full data (no holdout)

cooc_full, pop_counts_full = build_covisit_model(
    data,
    max_items_per_user=80,
    max_neighbors=400
)

# Popularity list on full data
items_by_pop_full = [it for it, _ in pop_counts_full.most_common()]

In [ ]:
def recommend_popularity_full(user, k=20):
    seen = set(data[user])
    recs = []
    for it in items_by_pop_full:
        if it in seen:
            continue
        recs.append(it)
        if len(recs) == k:
            break
    return recs

def recommend_covisit_full(user, k=20):
    seen = set(data[user])
    scores = defaultdict(float)

    for i in seen:
        if i not in cooc_full:
            continue
        for j, c in cooc_full[i].items():
            if j in seen:
                continue
            scores[j] += c / math.sqrt(pop_counts_full[i] * pop_counts_full[j])

    if not scores:
        return recommend_popularity_full(user, k=k)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    recs = [it for it, _ in ranked[:k]]

    # pad if needed
    if len(recs) < k:
        extra = recommend_popularity_full(user, k=k + 50)
        for it in extra:
            if it not in seen and it not in recs:
                recs.append(it)
                if len(recs) == k:
                    break

    return recs

Algorithm 3: The third model we implemented is a Matrix Factorization approach using the Alternating Least Squares (ALS) algorithm for implicit feedback. In this method, each user and item is represented as a vector in a shared latent factor space, and recommendations are generated by ranking items whose latent vectors align most closely with the user’s vector. We constructed a sparse item–user interaction matrix from the training data and trained an ALS model to learn these latent factors. Unlike the popularity baseline, matrix factorization personalizes rankings by capturing higher-level patterns in user–item interactions, though it requires significantly more computation. When evaluated using leave-one-out NDCG@20 on a 500-user sample, the MF model achieved a score of 0.0248, outperforming randomness and the popularity baseline but falling short of the more direct co-visitation approach, which exploits dense item-item signals more effectively for this dataset.

In [ ]:
# Install implicit library (run once per Colab session)
!pip install implicit

from scipy.sparse import csr_matrix
import implicit

In [ ]:
# 10. Generate Submission File
# We use the Co-visitation model as it performed best (NDCG ~0.083)

output_file = "submission.txt"
print(f"Generating recommendations for {len(data)} users using Co-visitation (Full)...")

with open(output_file, "w") as f:
    for i, user in enumerate(data.keys()):
        # Get top 20 recs
        recs = recommend_covisit_full(user, k=20)

        # Format: User Item1 Item2 ...
        line = user + " " + " ".join(recs) + "\n"
        f.write(line)

        if (i + 1) % 5000 == 0:
            print(f"Processed {i + 1} users...")

print(f"Finished! Output saved to {output_file}")

In [ ]:
# 10. Generate Submission File
# We use the Co-visitation model as it performed best

output_file = "submission.txt"
print(f"Generating recommendations for {len(data)} users using Co-visitation (Full)...")

with open(output_file, "w") as f:
    for i, user in enumerate(data.keys()):
        # Get top 20 recs
        recs = recommend_covisit_full(user, k=20)

        # Format: User Item1 Item2 ...
        line = user + " " + " ".join(recs) + "\n"
        f.write(line)

        if (i + 1) % 10000 == 0:
            print(f"Processed {i + 1} users...")

print(f"Finished! Output saved to {output_file}")

In [ ]:
# Verify the output file
print(f"First 5 lines of {output_file}:")
with open(output_file, "r") as f:
    for _ in range(5):
        print(f.readline().strip())

In [ ]:
# 10. Generate Submission File
# We use the Co-visitation model as it performed best

output_file = "submission.txt"
print(f"Generating recommendations for {len(data)} users using Co-visitation (Full)...")

with open(output_file, "w") as f:
    for i, user in enumerate(data.keys()):
        # Get top 20 recs
        recs = recommend_covisit_full(user, k=20)

        # Format: User Item1 Item2 ...
        line = user + " " + " ".join(recs) + "\n"
        f.write(line)

        if (i + 1) % 10000 == 0:
            print(f"Processed {i + 1} users...")

print(f"Finished! Output saved to {output_file}")

In [ ]:
# Verify the output file content
print(f"Preview of {output_file} (first 5 lines):")
with open(output_file, "r") as f:
    for _ in range(5):
        print(f.readline().strip())

In [ ]:
# 8. Matrix Factorization on Full Data

# A. Re-index users and items based on the COMPLETE 'data' dictionary
# (We do this to ensure we capture every single item, including those that were held out)
full_users_list = list(data.keys())
user2idx_full = {u: i for i, u in enumerate(full_users_list)}
idx2user_full = {i: u for u, i in user2idx_full.items()}

full_item_set = set()
for items in data.values():
    full_item_set.update(items)

full_items_list = list(full_item_set)
item2idx_full = {it: i for i, it in enumerate(full_items_list)}
idx2item_full = {i: it for it, i in item2idx_full.items()}

n_users_full = len(full_users_list)
n_items_full = len(full_items_list)

print(f"Full Dataset Indices: {n_users_full} users, {n_items_full} items")

In [ ]:
# B. Build Interaction Matrix (Users x Items)
rows = []
cols = []
vals = []

for u, items in data.items():
    u_idx = user2idx_full[u]
    for it in items:
        i_idx = item2idx_full[it]
        rows.append(u_idx)
        cols.append(i_idx)
        vals.append(1.0)

interaction_matrix_full = csr_matrix((vals, (rows, cols)),
                                     shape=(n_users_full, n_items_full))

# C. Train ALS Model on Full Data
als_model_full = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=10,
    random_state=42
)

alpha = 40.0
confidence_full = (interaction_matrix_full * alpha).astype('float32')

als_model_full.fit(confidence_full)
print("ALS Full Model trained.")

In [ ]:
# D. Define Full Recommendation Function
def recommend_mf_full(user, k=20):
    if user not in user2idx_full:
        return recommend_popularity_full(user, k=k)

    u_idx = user2idx_full[user]
    user_items = interaction_matrix_full[u_idx]

    # Recommend
    recs_idx, scores = als_model_full.recommend(
        userid=u_idx,
        user_items=user_items,
        N=k+50,
        filter_already_liked_items=True
    )

    # Map back to strings
    recs = []
    for i in recs_idx:
        recs.append(idx2item_full[i])

    # Pad if necessary
    if len(recs) < k:
        extra = recommend_popularity_full(user, k=k+50)
        seen = set(data[user])
        for it in extra:
            if it not in seen and it not in recs:
                recs.append(it)
                if len(recs) == k:
                    break
    return recs[:k]

In [ ]:
# 9. Verify all 3 models on a sample user

sample_u = list(data.keys())[0]
print(f"Recommendations for User {sample_u}:")

recs_pop = recommend_popularity_full(sample_u, k=5)
print(f"Popularity (Full):    {recs_pop}")

recs_co = recommend_covisit_full(sample_u, k=5)
print(f"Co-visitation (Full): {recs_co}")

recs_mf = recommend_mf_full(sample_u, k=5)
print(f"Matrix Fact. (Full):  {recs_mf}")

In [ ]:
# Build user and item index mappings from train_data

users_list = list(train_data.keys())
user2idx = {u: idx for idx, u in enumerate(users_list)}
idx2user = {idx: u for u, idx in user2idx.items()}

item_set = set()
for items in train_data.values():
    item_set.update(items)

items_list = list(item_set)
item2idx = {it: idx for idx, it in enumerate(items_list)}
idx2item = {idx: it for it, idx in item2idx.items()}

num_users = len(users_list)
num_items = len(items_list)

print("num_users:", num_users)
print("num_items:", num_items)

In [ ]:
# Build sparse interaction matrix: rows = users, cols = items
# implicit 0.7.x expects (n_users, n_items) for fit() and recommend()

rows = []
cols = []
vals = []

for u, items in train_data.items():
    u_idx = user2idx[u]
    for it in items:
        i_idx = item2idx[it]
        rows.append(u_idx)
        cols.append(i_idx)
        vals.append(1.0)  # implicit feedback

# Create CSR matrix of shape (num_users, num_items)
interaction_matrix = csr_matrix((vals, (rows, cols)),
                                shape=(num_users, num_items),
                                dtype=float)

print("Interaction matrix shape (users x items):", interaction_matrix.shape)
print("Non-zero entries:", interaction_matrix.nnz)

In [ ]:
# Train ALS matrix factorization model
# We re-initialize and train to ensure it matches the matrix shape

als_model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=10,
    random_state=42
)

# implicit works better if we weight the data (confidence)
alpha = 40.0
# confidence should be (users, items)
confidence = (interaction_matrix * alpha).astype('float32')

# Fit the model
als_model.fit(confidence)

print("ALS training complete.")

In [ ]:
def recommend_mf(user, k=20):
    """
    Recommend items for a user using ALS matrix factorization.
    Falls back to popularity if user not in mapping.
    """
    if user not in user2idx:
        return recommend_popularity(user, k=k)

    u_idx = user2idx[user]

    # implicit expects user_items to be a sparse matrix of shape (1, n_items)
    # for a single user recommendation.
    user_items = interaction_matrix[u_idx]

    n_recs = k + 50

    # userid is passed as an integer (scalar)
    # user_items must be the corresponding row
    recs_idx, scores = als_model.recommend(
        userid=u_idx,
        user_items=user_items,
        N=n_recs,
        filter_already_liked_items=True
    )

    rec_items = []
    for i_idx in recs_idx:
        it = idx2item[i_idx]
        rec_items.append(it)
        if len(rec_items) == k:
            break

    # Fallback padding
    if len(rec_items) < k:
        extra = recommend_popularity(user, k=k+50)
        seen = set(train_data[user])
        for it in extra:
            if it not in seen and it not in rec_items:
                rec_items.append(it)
                if len(rec_items) == k:
                    break

    return rec_items

In [ ]:
import random

users_list = list(train_data.keys())
random.seed(42)
sample_users_mf = random.sample(users_list, 500)  # start small to test

print("Evaluating MF model on 500 users...")
mf_ndcg_20 = mean_ndcg_at_k(recommend_mf, sample_users_mf, val_items, k=20)
print("MF (ALS) NDCG@20 on 500-user sample:", mf_ndcg_20)

print("\n" + "="*40)
print("FINAL MODEL COMPARISON (NDCG@20)")
print("="*40)
print(f"Popularity:    {pop_ndcg_20:.4f}")
print(f"Co-visitation: {covisit_ndcg_20:.4f}")
print(f"Matrix Fact.:  {mf_ndcg_20:.4f}")
print("="*40)